In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

PROCESSED_DIR = Path("./data/processed")


In [2]:
df = pd.read_csv(PROCESSED_DIR / "atp_matches.csv")
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2006-580,Australian Open,Hard,128.0,G,2006-01-16,103,103908,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,48.0,713.0,81.0,492.0
1,2006-580,Australian Open,Hard,128.0,G,2006-01-16,107,103990,16.0,NaN,...,50.0,35.0,20.0,14.0,9.0,13.0,19.0,1490.0,22.0,1390.0
2,2006-580,Australian Open,Hard,128.0,G,2006-01-16,104,103206,25.0,NaN,...,82.0,50.0,12.0,17.0,17.0,26.0,28.0,1080.0,9.0,2175.0
3,2006-580,Australian Open,Hard,128.0,G,2006-01-16,105,102148,NaN,NaN,...,81.0,51.0,26.0,23.0,5.0,13.0,65.0,580.0,10.0,2050.0
4,2006-580,Australian Open,Hard,128.0,G,2006-01-16,106,103970,11.0,NaN,...,57.0,31.0,14.0,13.0,7.0,13.0,14.0,1635.0,21.0,1390.0


In [3]:
## first we only work on the first 5 years to get our initial b-scores
initial_df = df[(df["tourney_date"] >= "2006-01-01") & (df["tourney_date"] < "2011-01-01") & (df["surface"] == "Grass"
"")].copy()

print(initial_df.shape)


(1278, 49)


In [4]:
t = pd.Timestamp("2010-12-31")
alpha_days = 365
initial_df["tourney_date"] = pd.to_datetime(initial_df["tourney_date"]) 

##following the logic of the paper on decay over time, can be tuned prob

initial_df["age_days"] = (t - initial_df["tourney_date"]).dt.days
initial_df["decay_weight"] = 1/ (1 + initial_df["age_days"] / alpha_days)

initial_df[["tourney_date", "age_days", "decay_weight"]].head()

,tourney_date,age_days,decay_weight
561,2006-06-12,1663,0.17998
562,2006-06-12,1663,0.17998
563,2006-06-12,1663,0.17998
564,2006-06-12,1663,0.17998
565,2006-06-12,1663,0.17998


In [5]:
##create the data frame for the edges to convert

edge_df = (initial_df
           .groupby(["loser_name", "winner_name"], as_index = False)["decay_weight"]
           .sum()
           .rename(columns={"loser_name": "source", "winner_name": "target", "decay_weight": "weight"}))

print(edge_df.head())
print(edge_df.shape)

             source                  target    weight
0  Adrian Mannarino            Marc Gicquel  0.395879
1   Agustin Calleri            Gilles Simon  0.283826
2   Agustin Calleri  Guillermo Garcia Lopez  0.395879
3   Agustin Calleri          Hyung Taik Lee  0.221212
4   Agustin Calleri           Stan Wawrinka  0.181231
(1214, 3)


In [6]:
print(edge_df.columns)

Index(['source', 'target', 'weight'], dtype='str')


In [7]:
##now the actual graph

G0 = nx.from_pandas_edgelist(
    edge_df,
    source = "source",
    target = "target",
    edge_attr = "weight",
    create_using = nx.DiGraph()
)

print(G0.number_of_nodes())
print(G0.number_of_edges())

292
1214


In [8]:
## simple check if all players are in the graph, should be 

all_players = pd.unique(
    pd.concat([initial_df["winner_name"], initial_df["loser_name"]], ignore_index=True)
)

G0.add_nodes_from(all_players)

print(len(all_players))
print(G0.number_of_nodes())

292
292


In [9]:
bscore_dict = nx.eigenvector_centrality(
    G0,
    max_iter=1000,
    tol = 1e-9,
    weight = "weight"
)

bscore_df = (
    pd.DataFrame({
        "player": list(bscore_dict.keys()),
        "bscore_2010": list(bscore_dict.values())
    })
    .sort_values("bscore_2010", ascending = False)
    .reset_index(drop = True)
)

bscore_df.head()

,player,bscore_2010
0,Roger Federer,0.435326
1,Rafael Nadal,0.407630
2,Andy Roddick,0.288661
3,Tomas Berdych,0.253761
4,Lleyton Hewitt,0.226290


In [10]:
## scores should be normalized in euclidean norm 1, meaning if all went correctly we get norm = 1.0,
#  at least ca.

norm = np.sqrt((bscore_df["bscore_2010"] **2).sum())
print(norm)

0.9999999999999999


In [11]:
matches_df = pd.read_csv(PROCESSED_DIR / "atp_matches.csv")

history_matches = matches_df[(matches_df["tourney_date"] >= "2006-01-01") & (matches_df["tourney_date"] < "2011-01-01") & (matches_df["surface"] == "Grass")].copy()
history_matches["tourney_date"] = pd.to_datetime(history_matches["tourney_date"])

future_matches = matches_df[(matches_df["tourney_date"] >= "2011-01-01") & (matches_df["surface"] == "Grass")].copy()
future_matches["tourney_date"] = pd.to_datetime(future_matches["tourney_date"])
print(history_matches.shape)
print(future_matches.shape)


(1278, 49)
(3290, 49)


In [12]:
def build_edge_df(match_history, t , alpha_days=365):
    df = match_history.copy()
    df["age_days"] = (t -df["tourney_date"]).dt.days
    df["decay_weight"] = 1/ (1 + df["age_days"] / alpha_days)
    edge_df = (df
           .groupby(["loser_name", "winner_name"], as_index = False)["decay_weight"]
           .sum()
           .rename(columns={"loser_name": "source", "winner_name": "target", "decay_weight": "weight"}))
    return edge_df

In [13]:
def build_graph(edge_df, all_players=None):
    G = nx.from_pandas_edgelist(
        edge_df,
        source = "source",
        target = "target",
        edge_attr = "weight",
        create_using = nx.DiGraph()
    )

    if all_players is not None:
        G.add_nodes_from(all_players)
        
    return G

In [14]:
def compute_bscore(G):
    bscore = nx.eigenvector_centrality(
        G,
        weight = "weight",
        max_iter=1000,
        tol=1e-9
    )
    return bscore

In [15]:
t0 = pd.Timestamp("2010-12-31")

all_players_0 = pd.unique(
    pd.concat(
        [history_matches["winner_name"], history_matches["loser_name"]],
        ignore_index=True

    )
)

edge_df_0 = build_edge_df(history_matches, t=t0, alpha_days=365)
G0 = build_graph(edge_df_0, all_players=all_players_0)
bscore_0 = compute_bscore(G0)

print(len(bscore_0))
print(list(bscore_0.items())[:5])


292
[('Adrian Mannarino', 2.9380057846803306e-17), ('Marc Gicquel', 0.02956909988560604), ('Agustin Calleri', 0.0007112319198760595), ('Gilles Simon', 0.011038101493078066), ('Guillermo Garcia Lopez', 0.09009732453741227)]


In [16]:
bscore_0_df = (
    pd.DataFrame({
        "player": list(bscore_0.keys()),
        "bscore": list(bscore_0.values())
    })
    .sort_values("bscore", ascending = False)
    .reset_index(drop = True)
)

default_bscore_0 = bscore_0_df["bscore"].quantile(0.25)

print("default_bscore_0:", default_bscore_0)
bscore_0_df.tail(10)

bscore_0_df.to_csv(PROCESSED_DIR / "bscore_0_grass.csv", index = False)

default_bscore_0: 2.9380057846803306e-17


In [17]:
##cant update day by day because not how dataset is structured

rolling_dates = sorted(future_matches["tourney_date"].unique())

print(len(rolling_dates))
print(rolling_dates[:5])
print(rolling_dates[-5:])

48
[Timestamp('2011-06-06 00:00:00'), Timestamp('2011-06-12 00:00:00'), Timestamp('2011-06-20 00:00:00'), Timestamp('2012-06-11 00:00:00'), Timestamp('2012-06-17 00:00:00')]
[Timestamp('2023-07-03 00:00:00'), Timestamp('2024-06-10 00:00:00'), Timestamp('2024-06-17 00:00:00'), Timestamp('2024-06-24 00:00:00'), Timestamp('2024-07-01 00:00:00')]


In [18]:
sorted(matches_df["round"].dropna().unique())
round_order = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}

In [19]:
history_matches["round_order"] = history_matches["round"].map(round_order)

future_matches["round_order"] = future_matches["round"].map(round_order)

history_matches

rolling_round = (
    future_matches[["tourney_date", "tourney_id", "round_order"]]
    .drop_duplicates()
    .sort_values(["tourney_date","tourney_id", "round_order"])
    .reset_index(drop=True)
)
rolling_round.head(20)

,tourney_date,tourney_id,round_order
0,2011-06-06,2011-311,2
1,2011-06-06,2011-311,3
2,2011-06-06,2011-311,4
3,2011-06-06,2011-311,5
4,2011-06-06,2011-311,6
5,2011-06-06,2011-311,7
6,2011-06-06,2011-500,3
7,2011-06-06,2011-500,4
8,2011-06-06,2011-500,5
9,2011-06-06,2011-500,6


In [20]:
future_matches[future_matches["round_order"].isna()]["round"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [21]:
current_history = history_matches.copy()
bscore_by_block = {}

enriched_blocks = []

for t_block, tourney_id, round_order in rolling_round.itertuples(index=False, name=None):
    block_matches = future_matches[
        (future_matches["tourney_id"] == tourney_id) &
        (future_matches["round_order"] == round_order)
        ].copy()
    
    t_block = block_matches["tourney_date"].iloc[0]

    all_players = pd.unique(
        pd.concat(
            [current_history["winner_name"], current_history["loser_name"]],
            ignore_index=True
        )
    )

    edge_df = build_edge_df(current_history, t=t_block, alpha_days=365)
    G = build_graph(edge_df, all_players=all_players)
    bscore_dict = compute_bscore(G)

    block_key = (tourney_id, round_order)
    bscore_by_block[block_key] = bscore_dict
    
    block_matches["winner_score_pre"] = block_matches["winner_name"].map(bscore_dict).fillna(default_bscore_0)
    block_matches["loser_score_pre"] = block_matches["loser_name"].map(bscore_dict).fillna(default_bscore_0)
    block_matches["bscore_diff"] = block_matches["winner_score_pre"] - block_matches["loser_score_pre"]

    enriched_blocks.append(block_matches) 

    current_history = pd.concat([current_history, block_matches], ignore_index=True)   

In [22]:
print(len(enriched_blocks))
print(enriched_blocks[0][["tourney_id", "round", "winner_name", "loser_name", "winner_score_pre", "loser_score_pre"]].head())

360
     tourney_id round       winner_name        loser_name  winner_score_pre  \
6408   2011-311   R64    Arnaud Clement      Benoit Paire      1.028588e-01   
6409   2011-311   R64   Michael Russell  Denis Gremelmayr      3.137901e-15   
6410   2011-311   R64  Janko Tipsarevic       Blaz Kavcic      1.229864e-01   
6411   2011-311   R64   Thomaz Bellucci   Grigor Dimitrov      1.950376e-03   
6412   2011-311   R64      Marsel Ilhan         Jan Hajek      2.218064e-14   

      loser_score_pre  
6408     2.938006e-17  
6409     2.122610e-16  
6410     2.122610e-16  
6411     2.100163e-03  
6412     2.122610e-16  


In [23]:
print(enriched_blocks[6][["tourney_id", "round", "winner_name", "loser_name", "winner_score_pre", "loser_score_pre"]].head())

     tourney_id round         winner_name         loser_name  \
6447   2011-500   R32       Tomas Berdych    Ruben Bemelmans   
6448   2011-500   R32         Jan Hernych  Sergiy Stakhovsky   
6449   2011-500   R32        Igor Andreev     Potito Starace   
6450   2011-500   R32      Viktor Troicki      Mischa Zverev   
6451   2011-500   R32  Philipp Petzschner    Dominik Meffert   

      winner_score_pre  loser_score_pre  
6447          0.152714     1.140266e-17  
6448          0.013250     3.443121e-02  
6449          0.003973     1.155484e-16  
6450          0.023915     4.450551e-02  
6451          0.039760     1.023196e-02  


In [24]:
final_matches_with_bscore = pd.concat(enriched_blocks, ignore_index=True)
final_matches_with_bscore.to_csv(PROCESSED_DIR / "atp_matches_with_bscore_Grass.csv", index=False)

In [25]:
final_matches_with_bscore[
    (final_matches_with_bscore["tourney_id"] == "2011-500") &
    (final_matches_with_bscore["winner_name"].str.contains("Berdych", na=False))
][["tourney_id", "round", "winner_name", "winner_score_pre"]]

,tourney_id,round,winner_name,winner_score_pre
55,2011-500,R32,Tomas Berdych,0.152714
77,2011-500,R16,Tomas Berdych,0.153220
80,2011-500,QF,Tomas Berdych,0.162148


In [26]:
rows = []

for block_key, bscore_dict in bscore_by_block.items():
    tourney_id, round_order = block_key

    for player, score in bscore_dict.items():
        rows.append({
            "tourney_id": tourney_id,
            "round_order": round_order,
            "player": player,
            "bscore_grass": score
        })

bscore_snapshots_df = pd.DataFrame(rows)

In [27]:
bscore_snapshots_df.to_csv("./data/processed/bscore_snapshots_grass.csv", index=False)